# Metacatalog on sky image

Matplotlib band overlays + interactive **SkyWidget + Bokeh** catalog map.

Launch with `pixi run jupyter lab`. **Run cells in order.**


In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import astropy.units as u
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.visualization import ImageNormalize, PercentileInterval, AsinhStretch
from bokeh.models import ColumnDataSource, HoverTool, TapTool
from bokeh.plotting import figure
from bokeh.transform import factor_cmap
from matplotlib import pyplot as plt

from astrowidget import SkyWidget
from astrowidget.wcs import adjust_wcs_for_array_stride

# --- paths ----------------------------------------------------------------
FITS_ROOT = Path("/fast/claw")
LST_HOUR = "01h"  # image LST hour to display
METACATALOG_CSV = Path(f"/fast/claw/metacatalog/metacatalog_{LST_HOUR}.csv")
BANDS = ("Full", "Red", "Green", "Blue")

# catalog display (applied per band in the matplotlib overlay)
FLUX_PERCENTILE = 90.0
MAX_PLOT_SOURCES = 3000

# reticle overlay
RETICLE_ARM_PIX = 14
RETICLE_GAP_FRAC = 0.35

# HiPS (same paths as source_review.ipynb)
HIPS_ROOT = Path("/lustre/pipeline/calibration/hips")
HIPS_BACKGROUND = HIPS_ROOT / "Blue_I_deep_Taper_Robust-0.75_Jan25.hips"
HIPS_HTTP_PREFIX = os.environ.get("OVRO_HIPS_HTTP_BASE", "/calibration/hips")
os.environ.setdefault("OVRO_HIPS_HTTP_BASE", HIPS_HTTP_PREFIX)
os.environ.setdefault("OVRO_HIPS_ROOT", str(HIPS_ROOT))
HIPS_BACKGROUND_PERCENTILE_LOW = 1.0
HIPS_BACKGROUND_PERCENTILE_HIGH = 99.0
BACKGROUND_CUT_MIN = None
BACKGROUND_CUT_MAX = None
BACKGROUND_OPACITY = 1.0

# SkyWidget radio overlay (mirrors source_review SourceReviewConfig defaults)
OVERLAY_MAX_SIZE = 1024
OVERLAY_COLORMAP = "magma"
OVERLAY_STRETCH = "log"
OVERLAY_OPACITY = 1.0
OVERLAY_PERCENTILE_LOW = 2.0
OVERLAY_PERCENTILE_HIGH = 98.0

# SkyWidget + Bokeh scatter
ORIGIN_BAND_COLORS = {
    "Full": "#22d3ee",
    "Blue": "#60a5fa",
    "Green": "#4ade80",
    "Red": "#f87171",
}
WIDGET_FOV_DEG = 25.0
FOCUS_FOV_DEG = 8.0
BOKEH_MAP_PX = 840


In [ ]:

_OVERLAY_BAND_CANON = {b.lower(): b for b in BANDS}
_FITS_OVERLAY_RE = re.compile(
    r"^I_(?P<lst>\d+h)_.*_(?P<band>Full|Red|Green|Blue)\.fits$",
    re.IGNORECASE,
)


def discover_fits_overlays(root: Path = FITS_ROOT) -> list[dict[str, object]]:
    """Scan ``FITS_ROOT`` for per-LST band images usable as SkyWidget overlays."""
    found: list[dict[str, object]] = []
    for path in sorted(root.glob("I_*h_*_*.fits")):
        match = _FITS_OVERLAY_RE.match(path.name)
        if match is None:
            continue
        band = _OVERLAY_BAND_CANON.get(match.group("band").lower())
        if band is None:
            continue
        lst = match.group("lst").lower()
        found.append(
            {
                "lst": lst,
                "band": band,
                "path": path,
                "label": f"{lst} — {band}",
            }
        )
    if not found:
        msg = f"No overlay FITS under {root}/I_*h_*_{{Full,Red,Green,Blue}}.fits"
        raise FileNotFoundError(msg)
    return found


def load_fits_overlay(path: Path) -> tuple[np.ndarray, WCS]:
    """Load one FITS image + celestial WCS (memmap-friendly)."""
    with fits.open(path, memmap=True) as hdul:
        raw = np.squeeze(np.asarray(hdul[0].data, dtype=np.float32))
        header = hdul[0].header.copy()
        wcs_obj = WCS(header).celestial
    return np.where(np.isfinite(raw), raw, np.nan), wcs_obj


def prepare_strided_overlay(
    raw: np.ndarray,
    wcs_native: WCS,
    *,
    max_size: int = OVERLAY_MAX_SIZE,
) -> tuple[np.ndarray, WCS, int, int]:
    """Downsample for SkyWidget comm + adjust WCS (PreloadedCube stride pattern)."""
    display_raw = np.where(np.isfinite(raw), raw, 0.0).astype(np.float32)
    n_l, n_m = display_raw.shape
    stride_l = max(1, n_l // max_size)
    stride_m = max(1, n_m // max_size)
    display_data = display_raw[::stride_l, ::stride_m]
    display_wcs = adjust_wcs_for_array_stride(wcs_native, stride_l, stride_m)
    return display_data, display_wcs, stride_l, stride_m


OVERLAY_SOURCES = discover_fits_overlays()
print(f"SkyWidget overlay catalog: {len(OVERLAY_SOURCES)} FITS files")

def resolve_fits(pattern: str, root: Path = FITS_ROOT) -> Path:
    matches = sorted(root.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected 1 match for {pattern!r}, got {len(matches)}")
    return matches[0]


if not METACATALOG_CSV.is_file():
    raise FileNotFoundError(f"Missing per-LST metacatalog: {METACATALOG_CSV}")

meta = pd.read_csv(METACATALOG_CSV)
meta = meta[np.isfinite(meta["RA"]) & np.isfinite(meta["DEC"])]

images: dict[str, dict[str, object]] = {}
for band in BANDS:
    band_path = resolve_fits(f"I_{LST_HOUR}_*_{band}.fits")
    with fits.open(band_path, memmap=True) as hdul:
        band_data = np.squeeze(np.asarray(hdul[0].data, dtype=np.float32))
        header = hdul[0].header.copy()
        band_wcs = WCS(header).celestial
    band_data = np.where(np.isfinite(band_data), band_data, np.nan)
    images[band] = {"data": band_data, "wcs": band_wcs, "path": band_path}

data = images["Full"]["data"]
wcs = images["Full"]["wcs"]

print(f"Catalog: {len(meta)} sources ({LST_HOUR})")
for band in BANDS:
    info = images[band]
    bw = info["wcs"]
    print(f"{band}: {info['path'].name} shape={info['data'].shape} CRVAL=({bw.wcs.crval[0]:.3f}, {bw.wcs.crval[1]:.3f})")


## 1. Matplotlib overlays


In [ ]:
%matplotlib inline


In [ ]:
plt.close("all")

from matplotlib.collections import LineCollection

BAND_FLUX = {"Full": "Peak_flux", "Blue": "Peak_flux_Blue", "Green": "Peak_flux_Green", "Red": "Peak_flux_Red"}
BAND_COORDS = {
    "Full": ("RA", "DEC"),
    "Blue": ("RA_Blue", "DEC_Blue"),
    "Green": ("RA_Green", "DEC_Green"),
    "Red": ("RA_Red", "DEC_Red"),
}


def filter_band_catalog(catalog: pd.DataFrame, band: str) -> pd.DataFrame:
    ra_col, dec_col = BAND_COORDS[band]
    flux_col = BAND_FLUX[band]
    band_meta = catalog.loc[catalog["origin_band"] == band].copy()
    band_meta = band_meta[np.isfinite(band_meta[ra_col]) & np.isfinite(band_meta[dec_col])]
    if FLUX_PERCENTILE is not None and len(band_meta):
        cutoff = np.nanpercentile(band_meta[flux_col], FLUX_PERCENTILE)
        band_meta = band_meta[band_meta[flux_col] >= cutoff]
    if len(band_meta) > MAX_PLOT_SOURCES:
        band_meta = band_meta.nlargest(MAX_PLOT_SOURCES, flux_col)
    return band_meta


def reticle_arm_deg(wcs_obj: WCS, dec_deg: float) -> tuple[float, float]:
    pix_scale_deg = float(np.mean(proj_plane_pixel_scales(wcs_obj)))
    arm_deg = RETICLE_ARM_PIX * pix_scale_deg
    cos_dec = max(float(np.cos(np.deg2rad(dec_deg))), 0.15)
    return arm_deg, arm_deg / cos_dec


def gap_crosshair_segments(ra_deg, dec_deg, *, wcs_obj: WCS, gap_frac: float):
    segments = []
    for ra, dec in zip(ra_deg, dec_deg, strict=True):
        arm_dec, arm_ra = reticle_arm_deg(wcs_obj, dec)
        gap_dec = arm_dec * gap_frac
        gap_ra = arm_ra * gap_frac
        segments.extend(
            [
                [(ra - arm_ra, dec), (ra - gap_ra, dec)],
                [(ra + gap_ra, dec), (ra + arm_ra, dec)],
                [(ra, dec - arm_dec), (ra, dec - gap_dec)],
                [(ra, dec + gap_dec), (ra, dec + arm_dec)],
            ]
        )
    return segments


def add_reticles(ax, wcs_obj: WCS, ra_deg, dec_deg) -> None:
    if len(ra_deg) == 0:
        return
    segments = gap_crosshair_segments(ra_deg, dec_deg, wcs_obj=wcs_obj, gap_frac=RETICLE_GAP_FRAC)
    world = ax.get_transform("world")
    ax.add_collection(LineCollection(segments, transform=world, colors="black", linewidths=2.0, alpha=0.9, zorder=10, capstyle="round"))
    ax.add_collection(LineCollection(segments, transform=world, colors="white", linewidths=1.2, alpha=1.0, zorder=11, capstyle="round"))


def plot_band_overlay(band: str) -> plt.Figure:
    info = images[band]
    band_data, band_wcs = info["data"], info["wcs"]
    band_meta = filter_band_catalog(meta, band)
    ra_col, dec_col = BAND_COORDS[band]
    norm = ImageNormalize(band_data, interval=PercentileInterval(99.5), stretch=AsinhStretch())
    fig = plt.figure(figsize=(9, 9))
    ax = fig.add_subplot(1, 1, 1, projection=band_wcs)
    ax.imshow(band_data, origin="lower", cmap="inferno", norm=norm)
    add_reticles(ax, band_wcs, band_meta[ra_col].to_numpy(), band_meta[dec_col].to_numpy())
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.grid(color="white", ls=":", alpha=0.25)
    fig.tight_layout()
    return fig


for band in BANDS:
    plot_band_overlay(band)


## 2. SkyWidget + Bokeh catalog (stacked)

Sky view and catalog map render **in one cell** — sky on top, Bokeh below. Tap a marker to focus the sky view.


In [ ]:
import ipywidgets as widgets
import panel as pn
from IPython.display import display

from scipy.ndimage import map_coordinates

from astrowidget.wcs import build_reproject_maps
from ovro_lwa_portal.viz.hips import compute_hips_percentile_cuts, hips_background_survey_url
from ovro_lwa_portal.viz.pipeline_qa_app import _patch_astrowidget_get_wcs

# Match source_review.ipynb: patch astrowidget WCS before creating SkyWidget.
_patch_astrowidget_get_wcs()

pn.extension("bokeh")



def _reproject_fits_for_shader(
    data: np.ndarray,
    wcs_obj: WCS,
    *,
    crval_ra: float,
    crval_dec: float,
) -> tuple[np.ndarray, WCS]:
    """Reproject a FITS C-order array onto the HiPS view tangent plane.

    Astropy ``hdul[0].data`` is ``(NAXIS2, NAXIS1)`` while astrowidget's
    ``apply_reproject_maps`` assumes numpy dim0 = WCS axis 1 (Zarr ``l``).
    Sample with ``[axis2, axis1]`` coordinates so catalog sources land on
    the same sky as HiPS after ``_push_image_frame``.
    """
    maps = build_reproject_maps(
        wcs_obj,
        data.shape,
        crval_ra=crval_ra,
        crval_dec=crval_dec,
    )
    out = map_coordinates(
        data.astype(np.float64, copy=False),
        [maps.src_m, maps.src_l],
        order=1,
        mode="constant",
        cval=np.nan,
    )
    out = out.astype(np.float32, copy=False)
    out[~maps.near_hemisphere] = np.nan
    return out, maps.wcs_out

# --- Overlay source selection (lazy load; many LST × band FITS on disk) ---
_overlay_state: dict[str, object] = {
    "data": None,
    "wcs": None,
    "lst": None,
    "band": None,
    "path": None,
}


def _apply_overlay_source(entry: dict[str, object]) -> None:
    raw, wcs_native = load_fits_overlay(entry["path"])
    display_data, display_wcs, stride_l, stride_m = prepare_strided_overlay(
        raw,
        wcs_native,
        max_size=OVERLAY_MAX_SIZE,
    )
    _overlay_state["data"] = display_data
    _overlay_state["wcs"] = display_wcs
    _overlay_state["lst"] = entry["lst"]
    _overlay_state["band"] = entry["band"]
    _overlay_state["path"] = entry["path"]
    n_l, n_m = raw.shape
    print(
        f"Overlay source: {entry['label']} ({entry['path'].name}) "
        f"display={display_data.shape} stride=({stride_l}, {stride_m}) from {n_l}×{n_m}"
    )
    if str(entry["lst"]) != str(LST_HOUR).lower():
        print(
            f"Note: catalog/Bokeh map is for LST {LST_HOUR}; "
            f"overlay is {entry['lst']} — source positions may differ."
        )


def _default_overlay_index() -> int:
    target_lst = str(LST_HOUR).lower()
    for idx, entry in enumerate(OVERLAY_SOURCES):
        if entry["lst"] == target_lst and entry["band"] == "Full":
            return idx
    return 0


hips_url = hips_background_survey_url(
    HIPS_BACKGROUND,
    hips_root=HIPS_ROOT,
    http_prefix=HIPS_HTTP_PREFIX,
)

sky = SkyWidget()
sky.background_survey = hips_url
sky.show_grid = True
sky.invert_horizontal_pan = True
sky.background_opacity = float(BACKGROUND_OPACITY)
sky.colormap = OVERLAY_COLORMAP
sky.stretch = OVERLAY_STRETCH
sky.opacity = float(OVERLAY_OPACITY)

if BACKGROUND_CUT_MIN is not None and BACKGROUND_CUT_MAX is not None:
    sky.background_cut_min = float(BACKGROUND_CUT_MIN)
    sky.background_cut_max = float(BACKGROUND_CUT_MAX)
else:
    try:
        cut_lo, cut_hi = compute_hips_percentile_cuts(
            HIPS_BACKGROUND,
            percentile_low=HIPS_BACKGROUND_PERCENTILE_LOW,
            percentile_high=HIPS_BACKGROUND_PERCENTILE_HIGH,
        )
        sky.background_cut_min = cut_lo
        sky.background_cut_max = cut_hi
    except (FileNotFoundError, ValueError) as exc:
        print(f"WARNING: HiPS cuts not set — {exc}")

print(f"HiPS background: {hips_url}")

center = SkyCoord(wcs.wcs.crval[0] * u.deg, wcs.wcs.crval[1] * u.deg)
sky.goto(center, fov=WIDGET_FOV_DEG * u.deg)

# View-locked overlay: reproject radio data to the current HiPS view after pan/zoom.
sky.overlay_view_lock = True

overlay_source_dropdown = widgets.Dropdown(
    options=[(entry["label"], idx) for idx, entry in enumerate(OVERLAY_SOURCES)],
    value=_default_overlay_index(),
    description="Overlay image:",
    layout=widgets.Layout(min_width="18em"),
)

overlay_toggle = widgets.Checkbox(
    value=False,
    description="Show overlay",
    indent=False,
)


def _push_fits_overlay_at_view(
    *,
    center: SkyCoord | None = None,
    fov: u.Quantity | None = None,
    update_view: bool = False,
) -> None:
    """Reproject Full-band FITS onto the active HiPS view tangent plane."""
    if center is None:
        center = sky.view_center_skycoord()
    ra_deg = float(center.icrs.ra.deg)
    dec_deg = float(center.icrs.dec.deg)
    reproj_data, reproj_wcs = _reproject_fits_for_shader(
        _overlay_state["data"],
        _overlay_state["wcs"],
        crval_ra=ra_deg,
        crval_dec=dec_deg,
    )
    sky._push_image_frame(
        reproj_data,
        reproj_wcs,
        center=center,
        fov=fov,
        update_view=update_view,
        percentile_low=OVERLAY_PERCENTILE_LOW,
        percentile_high=OVERLAY_PERCENTILE_HIGH,
    )


def set_radio_overlay(enabled: bool) -> None:
    if enabled:
        _push_fits_overlay_at_view(update_view=False)
    else:
        sky.clear_image()


def _on_view_gesture_revision(change) -> None:
    if change.get("type") != "change":
        return
    if overlay_toggle.value:
        _push_fits_overlay_at_view(update_view=False)


sky.observe(_on_view_gesture_revision, names="view_gesture_revision")
overlay_toggle.observe(lambda ch: set_radio_overlay(bool(ch["new"])), names="value")


def _on_overlay_source_change(change) -> None:
    if change.get("type") != "change":
        return
    _apply_overlay_source(OVERLAY_SOURCES[int(change["new"])])
    if overlay_toggle.value:
        _push_fits_overlay_at_view(update_view=False)


_apply_overlay_source(OVERLAY_SOURCES[_default_overlay_index()])
overlay_source_dropdown.observe(_on_overlay_source_change, names="value")


def focus_sky_widget(coord: SkyCoord, *, fov_deg: float = FOCUS_FOV_DEG) -> None:
    if overlay_toggle.value:
        _push_fits_overlay_at_view(
            center=coord,
            fov=fov_deg * u.deg,
            update_view=True,
        )
    else:
        sky.goto(coord, fov=fov_deg * u.deg)
    sky.set_crosshair(coord)

# --- NCP ZEA catalog map ----------------------------------------------------
_NCP_ZEA = WCS(naxis=2)
_NCP_ZEA.wcs.ctype = ["RA---ZEA", "DEC--ZEA"]
_NCP_ZEA.wcs.crval = [0.0, 90.0]
_NCP_ZEA.wcs.crpix = [180.0, 180.0]
_NCP_ZEA.wcs.cdelt = [-0.5, 0.5]
_NCP_ZEA.wcs.cunit = ["deg", "deg"]
_NCP_CENTER_XY = (180.0, 180.0)

CATALOG_COORDS = {
    "Full": ("RA", "DEC"),
    "Blue": ("RA_Blue", "DEC_Blue"),
    "Green": ("RA_Green", "DEC_Green"),
    "Red": ("RA_Red", "DEC_Red"),
}
BAND_FLUX_COL = {
    "Full": "Peak_flux",
    "Blue": "Peak_flux_Blue",
    "Green": "Peak_flux_Green",
    "Red": "Peak_flux_Red",
}


def catalog_radec_deg(row: pd.Series) -> tuple[float, float]:
    ra_col, dec_col = CATALOG_COORDS[str(row["origin_band"])]
    return float(row[ra_col]), float(row[dec_col])


def catalog_skycoord(row: pd.Series) -> SkyCoord:
    ra_deg, dec_deg = catalog_radec_deg(row)
    return SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")


def format_hms_dms(ra_deg: float, dec_deg: float) -> tuple[str, str]:
    coord = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    ra_s = coord.ra.to_string(unit=u.hour, sep=":", precision=1, pad=True)
    dec_s = coord.dec.to_string(unit=u.deg, sep=":", alwayssign=True, precision=1, pad=True)
    return ra_s, dec_s


def radec_to_ncp_zea(ra_deg, dec_deg):
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    x, y = _NCP_ZEA.all_world2pix(ra, dec, 0)
    return np.atleast_1d(np.asarray(x, dtype=float)), np.atleast_1d(np.asarray(y, dtype=float))


def ncp_zea_scalar(ra_deg: float, dec_deg: float) -> tuple[float, float]:
    x, y = radec_to_ncp_zea(ra_deg, dec_deg)
    return float(x[0]), float(y[0])


def ncp_zea_polyline(ra_deg, dec_deg):
    x, y = radec_to_ncp_zea(ra_deg, dec_deg)
    ok = np.isfinite(x) & np.isfinite(y)
    return x[ok].tolist(), y[ok].tolist()


def catalog_flux(row: pd.Series) -> float:
    return float(row[BAND_FLUX_COL[str(row["origin_band"])]])


def flux_marker_sizes_log(flux_values, *, min_size=5.0, size_ratio=3.0):
    flux = np.maximum(np.asarray(flux_values, dtype=float), np.finfo(float).tiny)
    log_f = np.log10(flux)
    log_min, log_max = float(log_f.min()), float(log_f.max())
    max_size = min_size * size_ratio
    if log_max <= log_min:
        return np.full(flux.shape, (min_size + max_size) / 2.0)
    t = (log_f - log_min) / (log_max - log_min)
    return min_size + t * (max_size - min_size)


plot_meta = meta.copy()
plot_meta["color"] = plot_meta["origin_band"].map(ORIGIN_BAND_COLORS)
plot_meta = plot_meta.dropna(subset=["origin_band", "color"])
cat_ra_dec = plot_meta.apply(catalog_radec_deg, axis=1, result_type="expand")
plot_meta["cat_ra"], plot_meta["cat_dec"] = cat_ra_dec[0], cat_ra_dec[1]
plot_meta["_flux"] = plot_meta.apply(catalog_flux, axis=1)
hms = plot_meta.apply(lambda r: format_hms_dms(r["cat_ra"], r["cat_dec"]), axis=1, result_type="expand")
plot_meta["ra_hms"], plot_meta["dec_dms"] = hms[0], hms[1]

ax, ay = radec_to_ncp_zea(plot_meta["cat_ra"].to_numpy(), plot_meta["cat_dec"].to_numpy())
finite = np.isfinite(ax) & np.isfinite(ay)
plot_meta = plot_meta.loc[finite].copy().reset_index(drop=True)
ax, ay = ax[finite], ay[finite]
marker_sizes = flux_marker_sizes_log(plot_meta["_flux"].to_numpy())

source = ColumnDataSource(
    data=dict(
        ax=ax.tolist(),
        ay=ay.tolist(),
        cat_ra=plot_meta["cat_ra"].tolist(),
        cat_dec=plot_meta["cat_dec"].tolist(),
        ra_hms=plot_meta["ra_hms"].tolist(),
        dec_dms=plot_meta["dec_dms"].tolist(),
        peak=plot_meta["_flux"].tolist(),
        origin_band=plot_meta["origin_band"].tolist(),
        marker_size=marker_sizes.tolist(),
    )
)

band_factors = [b for b in BANDS if b in plot_meta["origin_band"].values]
band_palette = [ORIGIN_BAND_COLORS[b] for b in band_factors]
cx, cy = _NCP_CENTER_XY
eq_x, eq_y = ncp_zea_scalar(0.0, 0.0)
_map_radius = float(np.hypot(eq_x - cx, eq_y - cy))

scatter_fig = figure(
    title=f"Metacatalog {LST_HOUR} — {len(plot_meta)} sources (tap to focus sky view above)",
    height=BOKEH_MAP_PX,
    width=BOKEH_MAP_PX,
    sizing_mode="fixed",
    tools="pan,wheel_zoom,box_zoom,reset,tap",
    match_aspect=True,
    background_fill_color="#111111",
)
_pad = 6.0
scatter_fig.x_range.start = cx - _map_radius - _pad
scatter_fig.x_range.end = cx + _map_radius + _pad
scatter_fig.y_range.start = cy - _map_radius - _pad
scatter_fig.y_range.end = cy + _map_radius + _pad

ra_circle = np.linspace(0.0, 360.0, 361)
horizon_x, horizon_y = ncp_zea_polyline(ra_circle, np.zeros(361))
meridian_xs, meridian_ys, parallel_xs, parallel_ys = [], [], [], []
for ra_hour in range(0, 24, 3):
    dec_line = np.linspace(0.0, 89.5, 120)
    xs, ys = ncp_zea_polyline(np.full(dec_line.shape, ra_hour * 15.0), dec_line)
    meridian_xs.append(xs)
    meridian_ys.append(ys)
for dec_line_val in (15, 30, 45, 60, 75):
    xs, ys = ncp_zea_polyline(np.linspace(0.0, 360.0, 361), np.full(361, dec_line_val))
    parallel_xs.append(xs)
    parallel_ys.append(ys)

scatter_fig.multi_line(
    [horizon_x] + meridian_xs + parallel_xs,
    [horizon_y] + meridian_ys + parallel_ys,
    line_color="#666666",
    line_width=0.8,
    line_alpha=0.55,
)

catalog_scatter = scatter_fig.scatter(
    "ax",
    "ay",
    source=source,
    size="marker_size",
    alpha=0.8,
    line_color="white",
    line_alpha=0.35,
    color=factor_cmap("origin_band", palette=band_palette, factors=band_factors),
    legend_group="origin_band",
)
scatter_fig.add_tools(HoverTool(renderers=[catalog_scatter], tooltips=[
    ("origin_band", "@origin_band"), ("RA", "@ra_hms"), ("Dec", "@dec_dms"), ("Peak flux", "@peak{0.2f}"),
]))
tap_tool = TapTool()
scatter_fig.add_tools(tap_tool)
scatter_fig.toolbar.active_tap = tap_tool


def on_bokeh_tap(event) -> None:
    if event.x is None or event.y is None:
        return
    dist = np.hypot(np.asarray(source.data["ax"]) - event.x, np.asarray(source.data["ay"]) - event.y)
    i = int(dist.argmin())
    if dist[i] > 8.0:
        return
    focus_sky_widget(
        SkyCoord(
            ra=float(source.data["cat_ra"][i]) * u.deg,
            dec=float(source.data["cat_dec"][i]) * u.deg,
            frame="icrs",
        )
    )


scatter_fig.on_event("tap", on_bokeh_tap)

# Stack sky + catalog in one cell: ipywidgets display first, Panel Bokeh directly below.
display(
    widgets.VBox(
        [widgets.HBox([overlay_toggle, overlay_source_dropdown]), sky],
        layout=widgets.Layout(width="100%", min_height="620px"),
    )
)

catalog_pane = pn.pane.Bokeh(
    scatter_fig,
    width=BOKEH_MAP_PX,
    height=BOKEH_MAP_PX,
    sizing_mode="fixed",
)
pn.Column(catalog_pane, sizing_mode="stretch_width", margin=(0, 0, 0, 0))
